# Topic Ranking and Narrowing
<!-- structured-notebook -->
## Notebook Summary
Purpose: reduce the raw BERTopic output (several hundred topics) to a smaller set worth deep narrative and sentiment analysis, by combining a composite quality score with a vague-topic flagger.

Main steps:
- Compute per-topic features: doc count, sentiment statistics, lexical diversity, concreteness.
- Combine into a composite ranking score using the weights validated during the project.
- Flag candidate vague topics for subtopic decomposition (handled in `03_subtopic_analysis.ipynb`).
- Produce the shortlist of topics that feed `04_topic_matching/`.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/topics.csv` | Produced by preprocessing |
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/kept_metadata.parquet` | Produced by preprocessing |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/selected_for_subtopics.csv` | Consumed by `01_subtopic_analysis.ipynb` |


# Ranking Metric Weights
<!-- structured-notebook -->
## Summary
After exploring several ranking strategies (fixed doc-count threshold, score-curve inflection, hybrid review), the project settled on a composite score over five features:

| Metric | Weight | Purpose |
|---|---|---|
| `doc_count` | 0.40 | Reliability — too few docs means unstable sentiment/emotion estimates |
| `sentiment_std` | 0.20 | Polarized topics are more narratively interesting |
| `sentiment_polarity` | 0.20 | Extreme-valued topics surface rhetoric |
| `lexical_diversity` | 0.10 | Proxy for topic richness |
| `concreteness_score` | 0.10 | Grounded topics are easier to interpret |

Concreteness and diversity lexicons: Brysbaert et al. 2014 (https://link.springer.com/article/10.3758/s13428-013-0403-5#MOESM1).

The composite produced 239 ranked topics from the raw ~300. Manual labeling of the top ~50 was attempted but nearly every candidate passed inspection, so the cutoff shifted to a rule-based `is_vague` filter combined with the composite ranking. Final cut: top-78, then further narrowed by manual review of the top-20 during narrative/framing analysis.


# Ranking Script
<!-- structured-notebook -->
## Summary
Computes per-topic metrics, applies the weighted composite score, and writes a ranked CSV. This is the main ranker; `is_vague` and the top-50 marking script are downstream filters that consume its output.


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def plot_topic_elbow(scores, title="Topic Ranking Elbow Plot", top_k=None, annotate_elbow=True):
    """
    Plots a ranking curve of topics and helps identify an elbow point.

    Args:
        scores (list or np.ndarray): Scores (e.g., topic frequencies or coherence), sorted descending.
        title (str): Title for the plot.
        top_k (int, optional): Highlight up to this many topics on the plot.
        annotate_elbow (bool): Whether to annotate the elbow point.

    Returns:
        int: Index of the elbow (suggested cut-off), useful for selecting top-N topics.
    """
    scores = np.array(scores)
    x = np.arange(1, len(scores) + 1)

    # Normalize scores for elbow detection
    norm_x = (x - x.min()) / (x.max() - x.min())
    norm_y = (scores - scores.min()) / (scores.max() - scores.min())

    # Find the point with max distance from the straight line (first to last)
    line = np.vstack((norm_x, norm_y)).T
    start = line[0]
    end = line[-1]
    line_vec = end - start
    line_vec /= np.linalg.norm(line_vec)

    distances = np.cross(line - start, line_vec)
    elbow_idx = np.argmax(distances)
    elbow_score = scores[elbow_idx]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(x, scores, marker='o')
    if top_k:
        plt.axvline(top_k, color='orange', linestyle='--', label=f'top_k = {top_k}')
    if annotate_elbow:
        plt.axvline(elbow_idx + 1, color='red', linestyle='--', label=f'Elbow @ {elbow_idx + 1}')
        plt.scatter(elbow_idx + 1, elbow_score, color='red')
    plt.title(title)
    plt.xlabel("Topic Rank")
    plt.ylabel("Score")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return elbow_idx + 1  # return 1-based index for easier slicing

def normalize_scores(scores, method="log"):
    scores = np.array(scores)
    if method == "log":
        return np.log1p(scores)  # log(1 + x)
    elif method == "percent":
        return scores / scores[0]  # divide by max (assuming descending)
    else:
        return scores




def select_subtopic_candidates(
    df: pd.DataFrame,
    doc_count_col: str = "doc_count",
    sentiment_std_col: str = "std_sentiment",
    min_doc_threshold: int = 2500,
    high_std_threshold: float = 0.5,
    top_n: int = 50,
    output_path: str = "selected_for_subtopics.csv"
) -> pd.DataFrame:
    """
    Selects topics for subtopic modeling using a hybrid strategy:
    - Include top N topics by ranking
    - Include low-doc topics with high sentiment variability

    Parameters:
        df (pd.DataFrame): Input topic summary DataFrame.
        doc_count_col (str): Column name for document count.
        sentiment_std_col (str): Column name for sentiment std.
        min_doc_threshold (int): Minimum doc count to include by default.
        high_std_threshold (float): Sentiment std threshold to include low-count topics.
        top_n (int): Always include top N topics from input.
        output_path (str): File to save selected topic IDs.

    Returns:
        pd.DataFrame: Filtered DataFrame with selected topics.
    """
    # Always keep the top-N topics
    top_n_df = df.head(top_n)

    # Include low-count topics with high variability
    low_doc_high_std = df[
        (df[doc_count_col] < min_doc_threshold) &
        (df[sentiment_std_col] > high_std_threshold)
    ]

    # Combine and remove duplicates
    combined = pd.concat([top_n_df, low_doc_high_std]).drop_duplicates(subset=["topic"])

    # Save and return
    combined.to_csv(output_path, index=False)
    print(f"✅ Saved {len(combined)} selected topics for subtopic analysis to {output_path}")
    return combined

# Load ranking scores for topics
# rankings_df = pd.read_csv("topic_narrowing-saved_files/initial-rankings/topic_sentiment_ranked.csv")
#
# # Example: frequency or coherence scores of topics (sorted descending)
# topic_scores = rankings_df['topic_score']
#
# raw_elbow = plot_topic_elbow(topic_scores, title="Raw Topic Scores")
#
# smoothed = normalize_scores(topic_scores, method="log")
# smoothed_elbow = plot_topic_elbow(smoothed, title="Smoothed Topic Scores (log)")
#
# print(f"Suggested cut-off: top {raw_elbow} topics")
# print(f"Suggested cut-off after smoothing: top {smoothed_elbow} topics")

ranks = pd.read_csv("topic_narrowing-saved_files/initial-rankings/topic_sentiment_ranked.csv")
combed = select_subtopic_candidates(ranks)

# Vague-Topic Flagger
<!-- structured-notebook -->
## Summary
A topic is marked for subtopic decomposition if at least two of the following hold:

1. Doc count ≥ 2000
2. Sentiment std > 0.5
3. Generic keywords (e.g. `health`, `aging`, `people`)
4. Mixed or unclear document themes
5. High topic ranking score (from the composite metric above)

The function below implements these rules. Topics it flags go to `03_subtopic_analysis.ipynb` for hierarchical decomposition; topics it does not flag are kept as-is.


In [ ]:
import pandas as pd

# Simulated input: your topic summary from earlier
df = pd.read_csv("topic_narrowing-saved_files/augmented-rankings/topic_augmented_ranking.csv")

# Define heuristic thresholds
VAGUE_KEYWORDS = {
    "health", "wellness", "fitness", "life", "lifestyle", "supplement",
    "diet", "tip", "hack", "routine", "product", "thing", "way", "drug"
}

THRESH = {
    "kw_overlap_min": 2,          # ≥2 generic words in top_words
    "sent_std": 0.45,             # high sentiment spread
    "doc_lex_div": 0.50,          # low doc-level lexical diversity
    "sem_coh": 0.35,              # (optional) low semantic coherence -> vague/mixed
    "overlap_penalty": 0.40,      # (optional) heavy keyword overlap with other topics
    "abstractness": 0.60,         # (optional) 1 - normalized_concreteness >= 0.60
    "ambig_low": 0.30,            # (optional) normalized_avg_max_prob in [0.30, 0.60]
    "ambig_high": 0.60,
    "kw_uniqueness": 0.30         # (optional) low uniqueness -> more generic
}


tw = df["top_words"].fillna("").str.lower().str.split(", ")
kw_overlap = tw.apply(lambda kws: len(set(kws) & VAGUE_KEYWORDS))

# Core signals
vague_by_keywords = kw_overlap >= THRESH["kw_overlap_min"]
vague_by_sentiment = df["sentiment_std"].fillna(0) >= THRESH["sent_std"]
vague_by_diversity = df["doc_lexical_diversity"].fillna(1.0) <= THRESH["doc_lex_div"]

# Optional signals (only used if present)
vague_by_low_coh = (1 - df["normalized_semantic_coherence"]).ge(0)  # default False
if "normalized_semantic_coherence" in df.columns:
    # low coherence → more mixed → more likely vague
    vague_by_low_coh = (df["normalized_semantic_coherence"].fillna(1.0) <= THRESH["sem_coh"])

vague_by_overlap = pd.Series(False, index=df.index)
if "normalized_overlap_penalty" in df.columns:
    vague_by_overlap = df["normalized_overlap_penalty"].fillna(0) >= THRESH["overlap_penalty"]
elif "overlap_penalty" in df.columns:
    # if raw overlap_penalty is present but not normalized, heuristic cutoff still OK
    vague_by_overlap = df["overlap_penalty"].fillna(0) >= THRESH["overlap_penalty"]

vague_by_abstract = pd.Series(False, index=df.index)
if "normalized_concreteness" in df.columns:
    abstractness = 1 - df["normalized_concreteness"].fillna(0.5)
    vague_by_abstract = abstractness >= THRESH["abstractness"]

vague_by_ambiguity = pd.Series(False, index=df.index)
if "normalized_avg_max_prob" in df.columns:
    p = df["normalized_avg_max_prob"].fillna(0.5)
    vague_by_ambiguity = (p >= THRESH["ambig_low"]) & (p <= THRESH["ambig_high"])

vague_by_low_uniq = pd.Series(False, index=df.index)
if "normalized_keyword_uniqueness" in df.columns:
    vague_by_low_uniq = df["normalized_keyword_uniqueness"].fillna(1.0) <= THRESH["kw_uniqueness"]

# Final flag (OR of available signals)
signals = {
    "kw_generic": vague_by_keywords,
    "sent_spread": vague_by_sentiment,
    "low_doc_div": vague_by_diversity,
    "low_coherence": vague_by_low_coh,
    "overlap_penalty": vague_by_overlap,
    "abstract": vague_by_abstract,
    "ambiguous": vague_by_ambiguity,
    "low_uniqueness": vague_by_low_uniq,
}
signal_df = pd.DataFrame(signals)

df["is_vague"] = signal_df.any(axis=1)

# Optional: concise reason string for auditing
def reason_row(i):
    active = [name for name, col in signals.items() if col.iat[i]]
    return ", ".join(active) if active else ""

df["vague_reason"] = [reason_row(i) for i in range(len(df))]

# (Optional) quick sanity view: how many topics flagged by each signal
summary = signal_df.sum().sort_values(ascending=False)
print("Vague signal counts:\n", summary)

# Save output
out_path = "topic_narrowing-saved_files/tagged_topics_with_vagueness.csv"
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


# Top-50 Selection for Subtopic Analysis
<!-- structured-notebook -->
## Summary
Final filter that marks the top-50 ranked topics as candidates for deep analysis. Produces `selected_for_subtopics.csv`, which the subtopic notebook reads to decide which parent topics to decompose.


In [ ]:
import os
import webbrowser
import pandas as pd
from bertopic import BERTopic
import pickle
import textwrap
import json

# Load data and model
top50_df = pd.read_csv('topic_narrowing-saved_files/initial-rankings/topic_sentiment_ranked_top50.csv')
topic_model = BERTopic.load('../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model')

with open('../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_doc_map.pkl', "rb") as f:
    topic_doc_map = pickle.load(f)


def print_section_header(text):
    """Print a formatted section header"""
    print(f"\n{'=' * 80}")
    print(f" {text.upper()} ")
    print(f"{'=' * 80}")


def print_subsection_header(text):
    """Print a formatted subsection header"""
    print(f"\n{'-' * 40}")
    print(f" {text} ")
    print(f"{'-' * 40}")


def format_document(doc, max_length=300):
    """Format document preview with word wrapping"""
    preview = doc[:max_length] + "..." if len(doc) > max_length else doc
    return textwrap.fill(preview, 80)


# List to store topics marked for subtopic analysis
topics_for_subtopic_analysis = []

# Print summary of topics
print_section_header("Topic Analysis Summary")
print(f"Analyzing the top {len(top50_df['topic'].unique())} topics")

vis_dir = "../../visualizations"
os.makedirs(vis_dir, exist_ok=True)

# Analyze each topic
for idx, topic_id in enumerate(top50_df['topic'].unique(), 1):
    print_section_header(f"Topic {topic_id} (#{idx} of {len(top50_df['topic'].unique())})")

    # Topic statistics
    topic_docs = topic_doc_map[topic_id]
    print(f"Number of documents: {len(topic_docs)}")

    # Keywords for the topic
    print_subsection_header("Top Keywords")
    keywords = topic_model.get_topic(topic_id)
    for word, score in keywords[:10]:  # Show top 10 keywords
        print(f"  • {word} ({score:.4f})")

    # Topic sentiment information
    topic_info = top50_df[top50_df['topic'] == topic_id].iloc[0]
    print_subsection_header("Sentiment Analysis")
    print(f"  • Mean Sentiment: {topic_info['mean_sentiment']}")
    print(f"  • Standard Deviation: {topic_info['std_sentiment']}")

    # Document examples
    print_subsection_header("Document Examples")
    for i, doc in enumerate(topic_docs[:3], 1):  # Limit to 3 examples
        print(f"\nDocument {i}:")
        print(format_document(doc))

    # Visualization
    # print_subsection_header("Topic Visualization")
    # fig_map = topic_model.visualize_topics()
    # map_path = os.path.join(vis_dir, f"topic_{topic_id}_map.html")
    # fig_map.write_html(map_path)
    #
    # print("Opening topic map visualization in browser...")
    # webbrowser.open("file://" + os.path.abspath(map_path))ebbrowser.open("file://" + os.path.abspath(map_path))

    # Prompt for subtopic analysis
    response = input("\nMark this topic for subtopic analysis? (y/n): ").strip().lower()
    if response in ['y', 'yes']:
        topics_for_subtopic_analysis.append(int(topic_id))  # Convert to standard Python int
        print(f"✓ Topic {topic_id} marked for subtopic analysis")
    else:
        print(f"✗ Topic {topic_id} not marked for subtopic analysis")

    # Prompt to continue
    if idx < len(top50_df['topic'].unique()):
        input("\nPress Enter to continue to the next topic...")

# Save the list of topics for subtopic analysis
if topics_for_subtopic_analysis:
    output_file = "topic_narrowing-saved_files/topics_for_subtopic_analysis.json"
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w") as f:
        json.dump({"topics": topics_for_subtopic_analysis}, f, indent=2)

    print_section_header("Summary")
    print(f"Marked {len(topics_for_subtopic_analysis)} topics for subtopic analysis:")
    for topic_id in topics_for_subtopic_analysis:
        keywords = [word for word, _ in topic_model.get_topic(topic_id)[:5]]
        print(f"  • Topic {topic_id}: {', '.join(keywords)}")
    print(f"\nSaved to: {output_file}")
else:
    print_section_header("Summary")
    print("No topics were marked for subtopic analysis.")

# Sentiment-Based Topic Narrowing
<!-- structured-notebook -->
## Summary
A complementary narrowing pass that uses per-topic sentiment signals to flag low-quality clusters. This was used exploratorily; the published pipeline relies more heavily on the composite ranking + vague flagger above, but this script is kept because the sentiment-driven signal was informative during iteration.


In [ ]:
import gc
import os
import pickle
import pandas as pd
import numpy as np
import torch
from bertopic import BERTopic
from tqdm import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter


def analyze_topic_sentiment(
        model_path,
        topic_doc_map_path,
        concreteness_lexicon_path,
        output_path="topic_narrowing-saved_files/topic_sentiment_summary.csv",
        min_doc_count=1000,
        include_topic_zero=False,
        pos_threshold=0.05,
        neg_threshold=-0.05
):
    """
    Analyze sentiment, diversity, and concreteness metrics for topics in a BERTopic model.

    Parameters:
    -----------
    model_path : str
        Path to the saved BERTopic model
    topic_doc_map_path : str
        Path to the pickle file containing the topic-document mapping
    concreteness_lexicon_path : str
        Path to the CSV file containing concreteness scores
    output_path : str, optional
        Path where the output CSV will be saved
    min_doc_count : int, optional
        Minimum number of documents a topic must have to be included
    include_topic_zero : bool, optional
        Whether to include topic 0 (often the "miscellaneous" topic)
    pos_threshold : float, optional
        Threshold for positive sentiment classification
    neg_threshold : float, optional
        Threshold for negative sentiment classification

    Returns:
    --------
    pd.DataFrame
        DataFrame containing sentiment and other metrics for each topic
    """
    # Initialize sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()

    # Load Concreteness Lexicon
    lexicon_df = pd.read_csv(concreteness_lexicon_path)
    lexicon = dict(zip(lexicon_df['Word'].str.lower(), lexicon_df['Conc.M']))

    # Load previously saved model and topic-document mapping
    topic_model = BERTopic.load(model_path)
    with open(topic_doc_map_path, "rb") as f:
        topic_doc_map = pickle.load(f)

    # Filter topics with too few documents
    topic_freq = topic_model.get_topic_freq()
    valid_topic_ids = topic_freq[topic_freq['Count'] >= min_doc_count]['Topic'].tolist()

    # Optionally exclude topic 0
    if not include_topic_zero and 0 in valid_topic_ids:
        valid_topic_ids.remove(0)

    # Compute sentiment metrics per topic
    topic_stats = []

    for topic, docs in tqdm(topic_doc_map.items(), desc="Computing sentiment per topic"):
        if topic not in valid_topic_ids:
            continue

        sentiments = [analyzer.polarity_scores(doc)["compound"] for doc in docs]

        mean_sent = np.mean(sentiments)
        std_sent = np.std(sentiments)

        pos = sum(1 for s in sentiments if s >= pos_threshold)
        neg = sum(1 for s in sentiments if s <= neg_threshold)
        neu = len(sentiments) - pos - neg

        sentiment_dist = {
            "pos_ratio": pos / len(sentiments),
            "neg_ratio": neg / len(sentiments),
            "neu_ratio": neu / len(sentiments),
        }

        # Get top 10 keywords
        top_words = topic_model.get_topic(topic)
        top_words_str = ", ".join([word for word, _ in top_words[:10]])

        # Lexical diversity: number of unique words / total words
        words = [word for word, _ in top_words]
        key_lexical_diversity = len(set(words)) / len(words) if words else 0

        tokens = " ".join(docs).split()
        doc_lexical_diversity = len(set(tokens)) / len(tokens) if tokens else 0

        # Compute average concreteness
        words = [word for word, _ in top_words]
        scores = [lexicon[word] for word in words if word in lexicon]
        topic_concreteness = sum(scores) / len(scores) if scores else None

        topic_stats.append({
            "topic": topic,
            "doc_count": len(docs),
            "mean_sentiment": round(mean_sent, 4),
            "sentiment_std": round(std_sent, 4),
            **sentiment_dist,
            "top_words": top_words_str,
            "key_lexical_diversity": key_lexical_diversity,
            "doc_lexical_diversity": doc_lexical_diversity,
            "concreteness": topic_concreteness
        })

    # Create DataFrame and sort by relevant metrics
    df_topic_summary = pd.DataFrame(topic_stats)
    df_topic_summary = df_topic_summary.sort_values(
        by=["doc_count", "sentiment_std", "doc_lexical_diversity"],
        ascending=False
    )

    # Save to CSV
    df_topic_summary.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    return df_topic_summary


from sklearn.preprocessing import MinMaxScaler

def compute_topic_scores(
    df: pd.DataFrame,
    doc_count_col: str = "doc_count",
    sentiment_std_col: str = "sentiment_std",
    sentiment_pos_col: str = "sentiment_ratio_pos",
    sentiment_neg_col: str = "sentiment_ratio_neg",
    key_lexical_diversity_col: str = "key_lexical_diversity",
    doc_lexical_diversity_col: str = "doc_lexical_diversity",
    concreteness_col: str = "concreteness_score",
    output_csv: str = "topic_sentiment_ranked.csv",
    weights: dict = None
) -> pd.DataFrame:
    """
    Compute a composite topic score and save the ranked topics to a CSV file.

    Parameters:
        df (pd.DataFrame): Input DataFrame with required columns.
        doc_count_col (str): Column name for document count.
        sentiment_std_col (str): Column name for sentiment standard deviation.
        sentiment_pos_col (str): Column name for positive sentiment ratio.
        sentiment_neg_col (str): Column name for negative sentiment ratio.
        key_lexical_diversity_col (str): Column name for keyword lexical diversity.
        doc_lexical_diversity_col (str): Column name for document lexical diversity.
        concreteness_col (str): Column name for concreteness score.
        output_csv (str): File name for output CSV.
        weights (dict): Optional dictionary to override weights.

    Returns:
        pd.DataFrame: Ranked DataFrame with added `topic_score` column.
    """

    df = df.copy()

    # Compute polarity from sentiment ratio difference
    df["sentiment_polarity"] = (df[sentiment_pos_col] - df[sentiment_neg_col]).abs()

    # Normalize features
    features = {
        "doc_count": doc_count_col,
        "sentiment_std": sentiment_std_col,
        "sentiment_polarity": "sentiment_polarity",
        "key_lexical_diversity": key_lexical_diversity_col,
        "doc_lexical_diversity": doc_lexical_diversity_col,
        "concreteness_score": concreteness_col,
    }

    scaler = MinMaxScaler()
    norm_values = scaler.fit_transform(df[list(features.values())])
    for norm_name, col in zip(features.keys(), norm_values.T):
        df[f"normalized_{norm_name}"] = col

    # Default weights (can override via argument)
    if weights is None:
        weights = {
            "doc_count": 0.4,
            "sentiment_std": 0.2,
            "sentiment_polarity": 0.2,
            "key_lexical_diversity": 0,
            "doc_lexical_diversity": 0.1,
            "concreteness_score": 0.1  # will subtract from 1 for abstractness
        }

    # Compute final topic score
    df["topic_score"] = (
        weights["doc_count"] * df["normalized_doc_count"] +
        weights["sentiment_std"] * df["normalized_sentiment_std"] +
        weights["sentiment_polarity"] * df["normalized_sentiment_polarity"] +
        weights["key_lexical_diversity"] * df["normalized_key_lexical_diversity"] +
        weights["doc_lexical_diversity"] * df["normalized_doc_lexical_diversity"] +
        weights["concreteness_score"] * (1 - df["normalized_concreteness_score"])
    )

    # Sort and export
    df = df.sort_values("topic_score", ascending=False)
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved ranked topic list to {output_csv}")

    return df


def compute_augmented_topic_scores(df, topic_model, model_name="all-mpnet-base-v2",
                                   cache_dir="../../preprocessed_docs", top_n_words=15):
    # 1) Load docs & global frequencies (streaming counter)
    with open(f"{cache_dir}/preprocessed_reddit_docs.pkl", "rb") as f:
        all_docs = pickle.load(f)

    global_word_freqs = Counter()
    for doc in all_docs:
        global_word_freqs.update(doc.split())

    # 2) Load topic words dict (limit to top_n for coherence)
    with open(f"{cache_dir}/topic-words-dict.pkl", "rb") as f:
        topic_words_dict = pickle.load(f)
    topics_in_df = set(df["topic"])
    topic_words_dict = {
        t: (words[:top_n_words] if isinstance(words, list) else [])
        for t, words in topic_words_dict.items() if t in topics_in_df
    }

    # 3) topics/max_probs: load cache or compute once
    topics_npy = os.path.join(cache_dir, "topics.npy")
    maxprobs_npy = os.path.join(cache_dir, "max_probs.npy")

    if os.path.exists(topics_npy) and os.path.exists(maxprobs_npy):
        topics = np.load(topics_npy)
        max_probs = np.load(maxprobs_npy)
    else:
        topics, probs = topic_model.transform(all_docs)
        max_probs = probs.max(axis=1)
        np.save(topics_npy, topics)
        np.save(maxprobs_npy, max_probs)
        del probs; gc.collect()

    # 4) avg_max_prob per topic (using max_probs only)
    #    (assumes you still know doc→topic; if not, recompute mask per topic)
    avg_max_prob = {}
    unique_topics = np.unique(topics)
    for tid in unique_topics:
        m = (topics == tid)
        avg_max_prob[tid] = float(max_probs[m].mean()) if m.any() else np.nan
    df["avg_max_prob"] = df["topic"].map(avg_max_prob)

    # 5) overlap penalty over top-N words from the model (defensive against None)
    top_n = 10
    topic_words = {}
    for t in df["topic"]:
        tw = topic_model.get_topic(t)
        if tw is None:
            topic_words[t] = []
        else:
            topic_words[t] = [w for w, _ in tw[:top_n]]

    topics_list = list(topic_words.keys())
    overlap_scores = {}
    for t in topics_list:
        words_t = set(topic_words[t])
        if not words_t:
            overlap_scores[t] = 0.0
            continue
        overlaps = 0
        for other in topics_list:
            if other == t:
                continue
            overlaps += len(words_t & set(topic_words[other]))
        denom = max(1, (len(topics_list) - 1) * top_n)
        overlap_scores[t] = overlaps / denom
    df["overlap_penalty"] = df["topic"].map(overlap_scores)

    # 6) sentiment polarity (fallback if no entropy later)
    df["sentiment_polarity"] = (df["pos_ratio"] - df["neg_ratio"]).abs()

    # 7) semantic coherence (SBERT on limited keyword sets)
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    st = SentenceTransformer(model_name, device=device)

    coherence_scores = {}
    for topic_id, words in tqdm(topic_words_dict.items(), desc="Computing semantic coherence"):
        if len(words) < 2:
            coherence_scores[topic_id] = 0.0
            continue
        emb = st.encode(words, convert_to_tensor=True)
        emb = emb.detach().cpu().numpy()
        sim = cosine_similarity(emb)
        np.fill_diagonal(sim, np.nan)
        coherence_scores[topic_id] = float(np.nanmean(sim[np.triu_indices_from(sim, k=1)]))
    df["semantic_coherence"] = df["topic"].map(coherence_scores)

    # 8) keyword uniqueness & novelty
    all_words = [w for words in topic_words_dict.values() for w in words]
    word_counts = pd.Series(all_words).value_counts() if all_words else pd.Series(dtype=int)

    uniqueness_scores = {
        tid: (np.mean([1 / word_counts.get(w, 1) for w in words]) if words else 0.0)
        for tid, words in topic_words_dict.items()
    }
    novelty_scores = {
        tid: (np.mean([1 / global_word_freqs.get(w, 1e-6) for w in words]) if words else 0.0)
        for tid, words in topic_words_dict.items()
    }
    df["keyword_uniqueness"] = df["topic"].map(uniqueness_scores)
    df["keyword_novelty"] = df["topic"].map(novelty_scores)

    # 9) normalize features
    columns = [
        "doc_count", "sentiment_std", "sentiment_polarity",
        "key_lexical_diversity", "doc_lexical_diversity", "concreteness",
        "semantic_coherence", "keyword_uniqueness", "keyword_novelty",
        "avg_max_prob", "overlap_penalty"
    ]
    # assert presence early to avoid KeyError
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in df: {missing}")

    scaler = MinMaxScaler()
    norm = scaler.fit_transform(df[columns].fillna(0))
    for i, col in enumerate(columns):
        df[f"normalized_{col}"] = norm[:, i]

    # log-scaled doc count
    x = np.log1p(df["doc_count"])
    df["normalized_log_doc_count"] = (x - x.min()) / (x.max() - x.min() + 1e-9)

    # sentiment entropy (fallback to polarity if not desired)
    p = np.clip(df[["pos_ratio", "neu_ratio", "neg_ratio"]].values, 1e-9, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    entropy = -(p * np.log(p)).sum(axis=1) / np.log(3)
    df["normalized_sentiment_entropy"] = (entropy - entropy.min()) / (entropy.max() - entropy.min() + 1e-9)

    inv_coherence = 1 - df["normalized_semantic_coherence"]
    abstractness  = 1 - df["normalized_concreteness"]
    moderate_uncertainty = 1 - (2 * (df["normalized_avg_max_prob"] - 0.5).abs())

    df["topic_score"] = (
        0.20 * df["normalized_log_doc_count"] +
        0.15 * df["normalized_sentiment_std"] +
        0.10 * df.get("normalized_sentiment_entropy", df["normalized_sentiment_polarity"]) +
        0.12 * df["normalized_doc_lexical_diversity"] +
        0.08 * df["normalized_key_lexical_diversity"] +
        0.12 * abstractness +
        0.12 * inv_coherence +
        0.06 * df["normalized_keyword_uniqueness"] +
        0.06 * df["normalized_keyword_novelty"] +
        0.05 * moderate_uncertainty -
        0.06 * df["normalized_overlap_penalty"]
    )

    df.sort_values("topic_score", ascending=False, inplace=True)
    return df


# Initial run to analyze topic sentiment
# results = analyze_topic_sentiment(
#     model_path="../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model",
#     topic_doc_map_path="../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_doc_map.pkl",
#     concreteness_lexicon_path=REDDIT_OUTPUT / "concreteness_scores.csv"
# )
#
# sentiment_df = pd.read_csv("topic_narrowing-saved_files/topic_sentiment_summary.csv")

# Compute topic scores and save ranked topics
# ranked_topics = compute_topic_scores(
#     sentiment_df,
#     doc_count_col="doc_count",
#     sentiment_std_col="mean_sentiment",
#     sentiment_pos_col="pos_ratio",
#     sentiment_neg_col="neg_ratio",
#     key_lexical_diversity_col="key_lexical_diversity",
#     doc_lexical_diversity_col="doc_lexical_diversity",
#     concreteness_col="concreteness",
#     output_csv="topic_sentiment_ranked.csv"
# )

# Top 50 (subtopics) and 20 (narrative framing) topics for further analysis
#sentiment_df = sentiment_df[:50]
#sentiment_df.to_csv("topic_narrowing-saved_files/topic_sentiment_ranked_top50.csv", index=False)

#sentiment_df = sentiment_df[:20]
#sentiment_df.to_csv("topic_narrowing-saved_files/topic_sentiment_ranked_top20.csv", index=False)

# Augmented topic ranking
df = pd.read_csv("topic_narrowing-saved_files/topic_sentiment_summary.csv")
topic_model = BERTopic.load("../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model")

# Compute scores
scored_df = compute_augmented_topic_scores(df, topic_model)

# Save ranked topics
scored_df.to_csv("topic_narrowing-saved_files/augmented-rankings/topic_augmented_ranking.csv", index=False)


---
<!-- nav-footer -->

← [02 topic merging](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/02_topic_merging.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [01 cross platform matching](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/01_cross_platform_matching.ipynb) →
